In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord, ICRS, Galactic
import astropy.units as u
import matplotlib as mpl

In [ ]:
dir = '/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth'


## Frequency cubes

In [ ]:
hdu = fits.open(dir+'/dragons/Q_dragons.fits')
dragonsQ = hdu[0].data
print(dragonsQ.shape)

hdu = fits.open(dir+'/dragons/U_dragons.fits')
dragonsU = hdu[0].data
print(dragonsU.shape)

#hdu = fits.open(dir+'/chime/Q_chime.fits')
hdu = fits.open('/srv/data/gmims/gmims-lbn/experiments/data/chime_regrid_Q.fits')
chimeQ = hdu[0].data
print(chimeQ.shape)

#hdu = fits.open(dir+'/chime/U_chime.fits')
hdu = fits.open('/srv/data/gmims/gmims-lbn/experiments/data/chime_regrid_U.fits')
chimeU = hdu[0].data
print(chimeU.shape)

#hdu = fits.open(dir+'/dragons_chime/Q_dragons_chime.fits')
hdu = fits.open('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/chime_dragons_Q_Sep26_highpass_mask.fits')
dragons_chimeQ = hdu[0].data
print(dragons_chimeQ.shape)

#hdu = fits.open(dir+'/dragons_chime/U_dragons_chime.fits')
hdu = fits.open('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/chime_dragons_U_Sep26_highpass_mask.fits')

dragons_chimeU = hdu[0].data
print(dragons_chimeU.shape)

hdu = fits.open('/srv/data/gmims/gmims-lbn/experiments/data/dragons_filtered_Q_Sep26_highpass_mask.fits')
dragonsQ_filt = hdu[0].data
print(dragonsQ_filt.shape)

hdu = fits.open('/srv/data/gmims/gmims-lbn/experiments/data/dragons_filtered_U_Sep26_highpass_mask.fits')
dragonsU_filt = hdu[0].data
print(dragonsU_filt.shape)

hdrQU = hdu[0].header
wcsQU = WCS(hdrQU)

## Dirty cubes

In [ ]:
hdu = fits.open(dir+'/dragons/FDF_tot_dirty.fits')
dragons = hdu[0].data
print(dragons.shape)

hdu = fits.open(dir+'/chime/FDF_tot_dirty.fits')
chime = hdu[0].data
print(chime.shape)

hdu = fits.open(dir+'/dragons_chime/FDF_tot_dirty.fits')
dragons_chime = hdu[0].data
print(dragons_chime.shape)

hdr = hdu[0].header
wcs = WCS(hdr)

#print(repr(hdr))

## Clean cubes

In [ ]:
hdu = fits.open(dir+'/dragons/FDF_clean_tot.fits')
dragons_clean = hdu[0].data
print(dragons_clean.shape)

hdu = fits.open(dir+'/chime/FDF_clean_tot.fits')
chime_clean = hdu[0].data
print(chime_clean.shape)

hdu = fits.open(dir+'/dragons_chime/FDF_clean_tot.fits')
dragons_chime_clean = hdu[0].data
print(dragons_chime_clean.shape)

## Peak maps

In [ ]:
hdu = fits.open(dir+'/dragons/clean_phiPeakPIfit_rm2.fits')
dragons_peak = hdu[0].data
print(dragons_peak.shape)

hdu = fits.open(dir+'/chime/clean_phiPeakPIfit_rm2.fits')
chime_peak = hdu[0].data
print(chime_peak.shape)

hdu = fits.open(dir+'/dragons_chime/clean_phiPeakPIfit_rm2.fits')
dragons_chime_peak = hdu[0].data
print(dragons_chime_peak.shape)

## RMSF

In [ ]:
hdu = fits.open(dir+'/chime/RMSF_tot.fits')
RMSF = hdu[0].data
print(RMSF.shape)

hdr_RMSF = hdu[0].header


## Axis arrays

In [ ]:
RA  = wcs.all_pix2world(range(wcs.array_shape[2]),0,0,0)[0]
dec = wcs.all_pix2world(0,range(wcs.array_shape[1]),0,0)[1]
FD  = wcs.all_pix2world(0,0,range(wcs.array_shape[0]),0)[2]

freq = wcsQU.all_pix2world(0,0,range(wcsQU.array_shape[0]),0)[2]/1e6

print(RA)
print(dec)
print(freq)

dxy = dec[1]-dec[0]
print(dxy)

FD_RMSF  = wcs.all_pix2world(0,0,range(WCS(hdr_RMSF).array_shape[0]),0)[2]-150.5
print(FD_RMSF)


## Plots

In [ ]:
def plot_2panel_maps(map1,map2,wcs,labels,title,vmax=1):

    fig = plt.figure(figsize=(18,8))
    fs = 16

    plt.subplots_adjust(hspace=0.0,left=0.05,right=0.999,top=0.95,bottom=0.05,wspace=0)

    crange = SkyCoord([24,0], [12,86], frame=ICRS, unit=(u.hr, u.deg))

    ax1 = fig.add_subplot(211, projection=wcs.celestial)
    ax2 = fig.add_subplot(212, projection=wcs.celestial)

    im1 = ax1.imshow(map1,vmin=-vmax,vmax=vmax,cmap='RdBu_r')
    im2 = ax2.imshow(map2,vmin=-vmax,vmax=vmax,cmap='RdBu_r')

    axs = [ax1,ax2]
    ims = [im1,im2]

    for i in range(0,len(axs)):
        axs[i].set_ylim(wcs.world_to_pixel(crange)[1])
        axs[i].set_xlim(wcs.world_to_pixel(crange)[0])
        cbar = fig.colorbar(ims[i], ax=axs[i], shrink=0.82, pad=0.005, aspect=12)
        cbar.set_label(label=labels[i],fontsize=fs)
        cbar.ax.tick_params(labelsize=fs) 
        axs[i].set_xlabel(' ')
        axs[i].set_ylabel('Declination',fontsize=fs)
        axs[i].tick_params(axis='both',labelsize=fs)
    
    ax2.set_xlabel('Right Ascension',fontsize=fs)

    ax1.set_title(title,fontsize=fs+2)

    return

In [ ]:
def plot_3panel_maps(map1,map2,map3,wcs,labels,title,vmax=1):

    fig = plt.figure(figsize=(18,12))
    fs = 16

    plt.subplots_adjust(hspace=0.0,left=0.05,right=0.999,top=0.95,bottom=0.05,wspace=0)

    crange = SkyCoord([24,0], [12,86], frame=ICRS, unit=(u.hr, u.deg))

    ax1 = fig.add_subplot(311, projection=wcs.celestial)
    ax2 = fig.add_subplot(312, projection=wcs.celestial)
    ax3 = fig.add_subplot(313, projection=wcs.celestial)

    im1 = ax1.imshow(map1,vmin=-vmax,vmax=vmax,cmap='RdBu_r')
    im2 = ax2.imshow(map2,vmin=-vmax,vmax=vmax,cmap='RdBu_r')
    im3 = ax3.imshow(map3,vmin=-vmax,vmax=vmax,cmap='RdBu_r')

    axs = [ax1,ax2,ax3]
    ims = [im1,im2,im3]

    for i in range(0,len(axs)):
        axs[i].set_ylim(wcs.world_to_pixel(crange)[1])
        axs[i].set_xlim(wcs.world_to_pixel(crange)[0])
        cbar = fig.colorbar(ims[i], ax=axs[i], shrink=0.82, pad=0.005, aspect=12)
        cbar.set_label(label=labels[i],fontsize=fs)
        cbar.ax.tick_params(labelsize=fs) 
        axs[i].set_xlabel(' ')
        axs[i].set_ylabel('Declination',fontsize=fs)
        axs[i].tick_params(axis='both',labelsize=fs)
    
    ax3.set_xlabel('Right Ascension',fontsize=fs)

    ax1.set_title(title,fontsize=fs+2)

    return

In [ ]:
def FD_moments(data,peakthresh,maxFD,RM_arr,M0=True,M1=True,M2=False,*args,**kwargs):
 
    dFD = abs(RM_arr[1]-RM_arr[0])

    # cut out FD range chosen:
    data_use = data[(abs(RM_arr) <= maxFD),:,:]    
    RM_arr_use = RM_arr[(abs(RM_arr) <= maxFD)]  

    # set any data points below PI threshold to NaN:
    data_use[data_use < peakthresh] = np.nan

    moments = {}
    if M0:
        M0_data = dFD*np.nansum(data_use,axis=0)
        M0_data[M0_data == 0] = np.nan
        moments['M0'] = M0_data
    if M1:
        M1_data = dFD*np.nansum(data_use*RM_arr_use[:,np.newaxis,np.newaxis],axis=0)/M0_data
        moments['M1'] = M1_data
    if M2:
        M2_data = np.sqrt(dFD*np.nansum(data_use*(RM_arr_use[:,np.newaxis,np.newaxis]-M1_data)**2,axis=0)/M0_data)
        moments['M2'] = M2_data
        
    return(moments)

In [ ]:
hdu = fits.open('/srv/data/dva/dragons_hoard/binned_channels_August2024_maps/Aug16_2024_all_freqs_BB/merged_maps/high/q/After_Ground_Iono_408MHz_high_58.q.fits')
Q408 = hdu[0].data

hdu = fits.open('/srv/data/dva/dragons_hoard/binned_channels_August2024_maps/Aug16_2024_all_freqs_BB/merged_maps/high/u/After_Ground_Iono_408MHz_high_58.u.fits')
U408 = hdu[0].data

wcs408 = WCS(hdu[0].header)

labels = [r'Stokes Q (K)',r'Stokes U (K)']

# Hack to get these into K:
plot_2panel_maps(Q408*0.06,U408*0.06,wcs408,labels,'DRAGONS maps',vmax=1.5)

plt.savefig('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/maps/dragons_QU_408.png')


In [ ]:
freq_plot = 612.5
fidx = abs(freq_plot - freq).argmin()
print(freq[fidx])

#labels = [r'Stokes Q (Jy beam$^{-1}$)',r'Stokes U (Jy beam$^{-1}$)']
labels = [r'Stokes Q (K)',r'Stokes U (K)']

#plot_2panel_maps(dragonsQ[fidx],dragonsU[fidx],wcsQU.dropaxis(2),labels,'DRAGONS maps')
#plt.savefig('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/maps/dragons_QU_612.png')

#plot_2panel_maps(dragonsQ_filt[fidx],dragonsU_filt[fidx],wcsQU.dropaxis(2),labels,'DRAGONS high-pass filtered maps')
#plt.savefig('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/maps/dragons_highpassfilt_QU_612.png')

#plot_2panel_maps(chimeQ[fidx],chimeU[fidx],wcsQU.dropaxis(2),labels,'CHIME maps')
#plt.savefig('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/maps/chime_QU_612.png')

plot_2panel_maps(dragons_chimeQ[fidx],dragons_chimeU[fidx],wcsQU.dropaxis(2),labels,'DRAGONS+CHIME maps')
plt.savefig('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/maps/dragons_chime_QU_612.png')

In [ ]:
labels = [r'rad m$^{-2}$',r'rad m$^{-2}$',r'rad m$^{-2}$']
plot_3panel_maps(dragons_peak, chime_peak, dragons_chime_peak,wcsQU.dropaxis(2),labels,'Peak Faraday depth maps',vmax=30)
plt.savefig('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/maps/peak_FD.png')

## Moments

In [ ]:
M_chime = FD_moments(chime_clean,0.0,80,FD,M0=True,M1=True,M2=False)
M_dragons = FD_moments(dragons_clean,0.0,80,FD,M0=True,M1=True,M2=False)
M_dragons_chime = FD_moments(dragons_chime_clean,0.0,80,FD,M0=True,M1=True,M2=False)


In [ ]:
labels = [r'rad m$^{-2}$',r'rad m$^{-2}$',r'rad m$^{-2}$']
plot_3panel_maps(M_dragons['M1'], M_chime['M1'], M_dragons_chime['M1'],wcsQU.dropaxis(2),labels,'Faraday depth M1',vmax=20)
plt.savefig('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/maps/M1.png')

In [ ]:
def zoom_plots(crange,map1,map2,map3,vmax=1):

    fig = plt.figure(figsize=(20,6))
    fs = 16
        
    plt.subplots_adjust(hspace=0.0)
        
    ax_chime   = fig.add_subplot(131, projection=wcs.dropaxis(2).celestial)
    ax_dragons = fig.add_subplot(132, projection=wcs.dropaxis(2).celestial)
    ax_combine = fig.add_subplot(133, projection=wcs.dropaxis(2).celestial)
    
    #freq_title = str(np.round(freq_list[fidx]/1e6,3))+' MHz'
    
    ax_chime.set_title('CHIME ',fontsize=fs)
    ax_dragons.set_title('DRAGONS ',fontsize=fs)
    ax_combine.set_title('DRAGONS+CHIME ',fontsize=fs)
    
    im_chime   = ax_chime.imshow(map1,vmin=-vmax,vmax=vmax,cmap='RdBu_r')
    im_dragons = ax_dragons.imshow(map2,vmin=-vmax,vmax=vmax,cmap='RdBu_r')
    im_combine = ax_combine.imshow(map3,vmin=-vmax,vmax=vmax,cmap='RdBu_r')
    
    axs = [ax_chime,ax_dragons,ax_combine]
    ims = [im_chime,im_dragons,im_combine]
        
    for i in range(0,len(axs)):
        axs[i].set_ylim(wcs.dropaxis(2).world_to_pixel(crange)[1])
        axs[i].set_xlim(wcs.dropaxis(2).world_to_pixel(crange)[0])
        #cbar = fig.colorbar(ims[i], ax=axs[i], shrink=0.65, pad=0.005, aspect=12)
        #cbar.set_label(label='K',fontsize=fs)
        #cbar.ax.tick_params(labelsize=fs) 
        axs[i].set_xlabel('RA',fontsize=fs)
        axs[i].set_ylabel(' ')
        axs[i].tick_params(axis='both',labelsize=fs)
    ax_chime.set_ylabel('dec',fontsize=fs)

    return

In [ ]:
crange = SkyCoord([6,2], [12,60], frame=ICRS, unit=(u.hr, u.deg))
zoom_plots(crange,M_dragons['M1'], M_chime['M1'], M_dragons_chime['M1'],vmax=15)

#plt.savefig('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/maps/M1_perTau.png')

In [ ]:
freq_plot = 612.5
fidx = abs(freq_plot - freq).argmin()
print(freq[fidx])

crange = SkyCoord([6,2], [12,60], frame=ICRS, unit=(u.hr, u.deg))
zoom_plots(crange,dragonsU[fidx],chimeU[fidx],dragons_chimeU[fidx],vmax=1.5)

#plt.savefig('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/maps/612U_perTau.png')

In [ ]:
freq_plot = 612.5
fidx = abs(freq_plot - freq).argmin()
print(freq[fidx])

crange = SkyCoord([7,3], [-15,20], frame=ICRS, unit=(u.hr, u.deg))
zoom_plots(crange,dragonsU[fidx],chimeU[fidx],dragons_chimeU[fidx],vmax=1.5)

## Make figures for feathering

In [ ]:
def feather(image, dxy, Rlim, feather_type, freq):
    
    print('Feathering for '+feather_type+' at '+str(freq)+'...')

    image[np.isnan(image)] = 0.0

    image_FFT = np.fft.fft2(image)
    image_FFT_shift = np.fft.fftshift(image_FFT)

    uv_freq_m = np.fft.fftshift(np.fft.fftfreq(image.shape[1]))/dxy
    uv_freq_n = np.fft.fftshift(np.fft.fftfreq(image.shape[0]))/dxy

    uv_m = 2*uv_freq_m*(3e8/(freq*1e6))*180/np.pi
    uv_n = 2*uv_freq_n*(3e8/(freq*1e6))*180/np.pi

    xuv, yuv = np.meshgrid(uv_m, uv_n)
    ruv = np.sqrt(xuv**2+yuv**2)
    
    b = np.pi/(Rlim[1]-Rlim[0])
    r0 = (Rlim[0]+Rlim[1])/2.

    if feather_type == 'AS':
        mask = 0.5*np.sin(b*(ruv - r0))+0.5
        mask[ruv > Rlim[1]] = 1
        mask[ruv < Rlim[0]] = 0
    if feather_type == 'SA':
        mask = -0.5*np.sin(b*(ruv - r0))+0.5
        mask[ruv > Rlim[1]] = 0
        mask[ruv < Rlim[0]] = 1

    fig, ax = plt.subplots(1,1,figsize=(8,5))
    ax.imshow(abs(image_FFT_shift),vmin=0,vmax=1e4,origin='lower',
                         extent=(uv_m[0],uv_m[-1],uv_n[0],uv_n[-1]))
    ax.set_xlim(-150,150)
    ax.set_ylim(-100,100)
    ax.set_xlabel('u (m)')
    ax.set_ylabel('v (m)')

    
    image_FFT_shift = image_FFT_shift*mask
    image_feath = np.fft.ifft2(np.fft.ifftshift(image_FFT_shift))
  
    return image_feath, mask, uv_m, uv_n

In [ ]:
hdu = fits.open('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/chime_K_Q.fits')
chime_KQ = hdu[0].data
print(chime_KQ.shape)

hdu = fits.open('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/chime_K_U.fits')
chime_KU = hdu[0].data
print(chime_KU.shape)

hdu = fits.open('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/dragons_deconv_Q_Sep26_highpass_mask.fits')
dragons_deconvQ = hdu[0].data
print(dragons_deconvQ.shape)

hdu = fits.open('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/dragons_deconv_U_Sep26_highpass_mask.fits')
dragons_deconvU = hdu[0].data
print(dragons_deconvU.shape)

In [ ]:
print(freq[fidx])

image_feath, mask, uv_m, uv_n = feather(dragonsU[fidx], dxy, [3,5], 'SA', freq[fidx])

#plt.savefig('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/maps/dragons_uv_StokesU.png')

In [ ]:
r = uv_m[uv_m>=0]

Rlim = [3,5]

b = np.pi/(Rlim[1]-Rlim[0])
r0 = (Rlim[0]+Rlim[1])/2.

mask_AS = 0.5*np.sin(b*(r - r0))+0.5
mask_AS[r > Rlim[1]] = 1
mask_AS[r < Rlim[0]] = 0

mask_SA = -0.5*np.sin(b*(r - r0))+0.5
mask_SA[r > Rlim[1]] = 0
mask_SA[r < Rlim[0]] = 1

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(5,3))

ax.plot(r,mask_AS,label='CHIME')
ax.plot(r,mask_SA,label='DRAGONS')

ax.set_xlim(0,20)
ax.set_xlabel(r'$\sqrt{u^2+v^2}$')
ax.legend()
plt.savefig('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/maps/uv_feathering.png')

In [ ]:
RA_plot  = 4.5*360/24
dec_plot = 35.0

ii = abs(RA  - RA_plot ).argmin()
jj = abs(dec - dec_plot).argmin()

print(RA[ii],dec[jj])

fig,ax = plt.subplots(1,1,figsize=(6,4))

#ax.plot(FD, chime[:,jj,ii],linestyle='dashed')
#ax.plot(FD, dragons[:,jj,ii],linestyle='dashed')
#ax.plot(FD, dragons_chime[:,jj,ii],linestyle='dashed')

ax.plot(FD, chime_clean[:,jj,ii],color='blue',label='CHIME')
ax.plot(FD, dragons_clean[:,jj,ii],color='red',label='DRAGONS')
ax.plot(FD, dragons_chime_clean[:,jj,ii],color='green',label='CHIME+DRAGONS')

#ax.axhline(y = 0.5, color='C0',linestyle='dashed')
#ax.axhline(y = 1.0, color='C1',linestyle='dashed')

ax.set_xlabel(r'rad m$^{-2}$')
ax.set_xlim(-80,80)
ax.grid()
ax.set_ylabel('Polarised intensity (K/RMSF)')
ax.legend()

ax2 = ax.twinx()
ax2.plot(FD_RMSF, RMSF[:,jj,ii],color='black',linestyle='dashed')
ax2.set_ylabel('RMSF')

plt.title('RA = '+str(RA_plot)+r'$^{\circ}$, dec = '+str(dec_plot)+r'$^{\circ}$')

plt.savefig('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/spectrum_'+str(RA_plot)+'_'+str(dec_plot)+'.png')

In [ ]:
2*360/24

In [ ]:
chime_med = np.nanmedian(chime,axis=(1,2))

In [ ]:
RA_plot  = 60
dec_plot = 60

ii = abs(RA  - RA_plot ).argmin()
jj = abs(dec - dec_plot).argmin()

print(RA[ii],dec[jj])

fig,ax = plt.subplots(1,1,figsize=(18,8))

ax.plot(FD, chime_med)
#ax.plot(FD, dragons[:,jj,ii])
#ax.plot(FD, dragons_chime[:,jj,ii])

ax.axhline(y = 0.5, color='C0',linestyle='dashed')
#ax.axhline(y = 1.0, color='C1',linestyle='dashed')

ax.set_xlim(-150,150)

ax2 = ax.twinx()
ax2.plot(FD_RMSF, RMSF[:,jj,ii],color='black')